# Fast Preview - Klasifikasi Penyakit Daun

Notebook ini adalah versi cepat untuk melihat contoh hasil awal tanpa menjalankan pipeline fitur berat. Dataset tetap memakai kelas sesuai proposal:
- Peach: healthy dan bacterial spot
- Pepper bell: healthy dan bacterial spot
- Strawberry: healthy dan leaf scorch

Strategi versi cepat:
- Mengambil sebagian kecil gambar per kelas.
- Resize gambar ke ukuran kecil.
- Ekstraksi fitur ringan dari warna HSV/RGB, estimasi area bercak, dan tekstur sederhana.
- Training Random Forest kecil agar hasil contoh cepat terlihat.

## 1. Setup Library

Cell instalasi disiapkan untuk environment yang belum punya package ML/plotting. Setelah instalasi, restart kernel jika import masih belum terbaca.

In [ ]:
# Jalankan hanya jika package belum tersedia.
# %pip install numpy pandas pillow matplotlib seaborn scikit-learn

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

from plant_classifier.model_store import load_model_bundle, save_model_bundle

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Konfigurasi Dataset Cepat

Nilai sampling dibuat kecil agar proses training dan evaluasi cepat. Jika hasil contoh sudah terlihat, angka sampling bisa dinaikkan perlahan untuk hasil yang lebih stabil.

In [ ]:
BASE_DIR = Path.cwd()
DATASET_ROOT = BASE_DIR / "archive" / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)"
TRAIN_DIR = DATASET_ROOT / "train"
VALID_DIR = DATASET_ROOT / "valid"
MODEL_DIR = BASE_DIR / "models"
FAST_MODEL_PATH = MODEL_DIR / "fast_random_forest.pickle"

SELECTED_CLASSES = {
    "Peach___Bacterial_spot": "Peach - Bacterial spot",
    "Peach___healthy": "Peach - Healthy",
    "Pepper,_bell___Bacterial_spot": "Pepper bell - Bacterial spot",
    "Pepper,_bell___healthy": "Pepper bell - Healthy",
    "Strawberry___Leaf_scorch": "Strawberry - Leaf scorch",
    "Strawberry___healthy": "Strawberry - Healthy",
}

# Sampling kecil supaya preview cepat.
TRAIN_PER_CLASS = 80
VALID_PER_CLASS = 20
IMAGE_SIZE = (96, 96)

print("Dataset root:", DATASET_ROOT)
print("Train folder ada:", TRAIN_DIR.exists())
print("Valid folder ada:", VALID_DIR.exists())
print("Model pickle fast:", FAST_MODEL_PATH)


In [ ]:
def collect_subset(split_dir, selected_classes, per_class):
    """Mengambil subset gambar secara acak dari tiap kelas."""
    rows = []
    valid_ext = {".jpg", ".jpeg", ".png", ".bmp"}

    for folder_name, label in selected_classes.items():
        class_dir = split_dir / folder_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Folder tidak ditemukan: {class_dir}")

        image_paths = [p for p in class_dir.iterdir() if p.suffix.lower() in valid_ext]
        image_paths = sorted(image_paths)
        sample_size = min(per_class, len(image_paths))
        sampled = random.sample(image_paths, sample_size)

        for image_path in sampled:
            rows.append({
                "path": image_path,
                "folder_class": folder_name,
                "label": label,
                "split": split_dir.name,
            })

    return pd.DataFrame(rows)

train_df = collect_subset(TRAIN_DIR, SELECTED_CLASSES, TRAIN_PER_CLASS)
valid_df = collect_subset(VALID_DIR, SELECTED_CLASSES, VALID_PER_CLASS)

print("Jumlah train preview:", len(train_df))
print("Jumlah valid preview:", len(valid_df))
display(pd.concat([train_df, valid_df]).groupby(["split", "label"]).size().reset_index(name="count"))

## 3. Cek Sampel Gambar

Visualisasi singkat ini memastikan subset yang dipakai sudah sesuai dan memberi gambaran awal perbedaan daun sehat dan berpenyakit.

In [ ]:
def show_samples(df, n_per_class=2):
    labels = sorted(df["label"].unique())
    fig, axes = plt.subplots(len(labels), n_per_class, figsize=(3.2 * n_per_class, 2.5 * len(labels)))
    axes = np.array(axes).reshape(len(labels), n_per_class)

    for row_idx, label in enumerate(labels):
        subset = df[df["label"] == label].sample(n_per_class, random_state=RANDOM_STATE)
        for col_idx, (_, row) in enumerate(subset.iterrows()):
            image = Image.open(row["path"]).convert("RGB")
            axes[row_idx, col_idx].imshow(image)
            axes[row_idx, col_idx].set_title(label, fontsize=9)
            axes[row_idx, col_idx].axis("off")

    plt.tight_layout()
    plt.show()

show_samples(train_df, n_per_class=2)

## 4. Ekstraksi Fitur Ringan

Versi cepat ini tidak memakai GLCM/LBP penuh. Fitur dibuat dari:
- Rata-rata dan standar deviasi channel RGB.
- Rata-rata dan standar deviasi channel HSV.
- Estimasi area daun dari saturation/value.
- Estimasi area bercak dari warna gelap, kuning, cokelat, atau kemerahan.
- Tekstur sederhana dari perubahan intensitas grayscale.

In [ ]:
def load_preview_image(image_path):
    """Membaca, resize, dan denoise ringan gambar."""
    image = Image.open(image_path).convert("RGB").resize(IMAGE_SIZE)

    # Median dan Gaussian blur ringan mengurangi noise kecil tanpa membuat proses berat.
    image = image.filter(ImageFilter.MedianFilter(size=3))
    image = image.filter(ImageFilter.GaussianBlur(radius=0.4))
    return image


def image_to_arrays(image):
    """Mengubah gambar PIL ke array RGB, HSV, dan grayscale."""
    rgb = np.asarray(image).astype(np.float32)
    hsv = np.asarray(image.convert("HSV")).astype(np.float32)
    gray = np.asarray(image.convert("L")).astype(np.float32)
    return rgb, hsv, gray


def make_leaf_and_spot_masks(hsv):
    """Membuat mask daun dan kandidat bercak dengan threshold sederhana."""
    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    # Area daun biasanya punya saturasi cukup jelas dan bukan background sangat gelap.
    leaf_mask = (s > 25) & (v > 35)

    # Kandidat bercak: area pada daun yang lebih gelap atau memiliki hue kuning/cokelat/kemerahan.
    reddish = (h < 18) | (h > 238)
    yellow_brown = (h > 18) & (h < 55)
    dark_spot = v < 120
    saturated_damage = s > 55
    spot_mask = leaf_mask & saturated_damage & (dark_spot | reddish | yellow_brown)

    return leaf_mask, spot_mask


def extract_fast_features(image_path):
    """Pipeline fitur cepat dari satu gambar."""
    image = load_preview_image(image_path)
    rgb, hsv, gray = image_to_arrays(image)
    leaf_mask, spot_mask = make_leaf_and_spot_masks(hsv)

    if leaf_mask.sum() == 0:
        leaf_mask = np.ones(leaf_mask.shape, dtype=bool)

    features = {}
    leaf_pixels = leaf_mask.sum()
    spot_pixels = spot_mask.sum()

    features["leaf_area_ratio"] = float(leaf_pixels / leaf_mask.size)
    features["spot_area_ratio"] = float(spot_pixels / max(leaf_pixels, 1))
    features["dark_leaf_ratio"] = float(((gray < 100) & leaf_mask).sum() / max(leaf_pixels, 1))

    for name, arr in [("rgb", rgb), ("hsv", hsv)]:
        for channel in range(3):
            values = arr[:, :, channel][leaf_mask]
            features[f"{name}_{channel}_mean"] = float(values.mean())
            features[f"{name}_{channel}_std"] = float(values.std())
            features[f"{name}_{channel}_p25"] = float(np.percentile(values, 25))
            features[f"{name}_{channel}_p75"] = float(np.percentile(values, 75))

    # Tekstur cepat: rata-rata perubahan intensitas horizontal dan vertikal.
    grad_y = np.abs(np.diff(gray, axis=0)).mean()
    grad_x = np.abs(np.diff(gray, axis=1)).mean()
    features["texture_grad_mean"] = float((grad_x + grad_y) / 2)
    features["texture_grad_x"] = float(grad_x)
    features["texture_grad_y"] = float(grad_y)

    # Histogram hue kasar membantu model membaca pola warna dominan.
    hue_values = hsv[:, :, 0][leaf_mask]
    hue_hist, _ = np.histogram(hue_values, bins=8, range=(0, 256), density=True)
    for idx, value in enumerate(hue_hist):
        features[f"hue_hist_{idx}"] = float(value)

    return features

In [ ]:
def build_fast_feature_table(df):
    """Mengekstrak fitur ringan untuk seluruh subset."""
    rows = []
    total = len(df)

    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        feature_row = extract_fast_features(row["path"])
        feature_row["path"] = str(row["path"])
        feature_row["label"] = row["label"]
        rows.append(feature_row)

        if idx % 100 == 0 or idx == total:
            print(f"Progress fitur: {idx}/{total}")

    return pd.DataFrame(rows)

train_features = build_fast_feature_table(train_df)
valid_features = build_fast_feature_table(valid_df)

display(train_features.head())
print("Jumlah fitur cepat:", train_features.drop(columns=["path", "label"]).shape[1])

## 5. Training Cepat Random Forest

Random Forest dipakai karena cukup kuat untuk fitur tabular dan tidak memerlukan preprocessing skala fitur yang rumit. Jumlah tree dibuat kecil agar preview cepat.

In [ ]:
feature_cols = [col for col in train_features.columns if col not in ["path", "label"]]

X_train = train_features[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
X_valid = valid_features[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_valid_label = valid_features["label"]

if FAST_MODEL_PATH.exists():
    bundle = load_model_bundle(FAST_MODEL_PATH)
    model = bundle.model
    encoder = bundle.label_encoder
    feature_cols = bundle.feature_columns
    X_train = train_features[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_valid = valid_features[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_train = encoder.transform(train_features["label"])
    y_valid = encoder.transform(y_valid_label)
    class_names = encoder.classes_
    pred_valid = model.predict(X_valid)
    print(f"Model fast dimuat dari pickle: {FAST_MODEL_PATH}")
else:
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(train_features["label"])
    y_valid = encoder.transform(y_valid_label)
    class_names = encoder.classes_

    model = RandomForestClassifier(
        n_estimators=80,
        max_depth=12,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)

    MODEL_DIR.mkdir(exist_ok=True)
    save_model_bundle(
        FAST_MODEL_PATH,
        variant="fast",
        model_name="Fast Random Forest",
        model=model,
        label_encoder=encoder,
        feature_columns=feature_cols,
        metrics={"accuracy": float(accuracy_score(y_valid, pred_valid))},
        metadata={"random_state": RANDOM_STATE, "train_per_class": TRAIN_PER_CLASS, "valid_per_class": VALID_PER_CLASS},
    )
    print(f"Model fast dilatih dan disimpan ke pickle: {FAST_MODEL_PATH}")


## 6. Evaluasi Cepat

Metrik di bawah hanya preview awal karena jumlah sampel dibuat kecil. Untuk laporan akhir, gunakan notebook lengkap atau naikkan jumlah sample per kelas.

In [ ]:
acc = accuracy_score(y_valid, pred_valid)
print(f"Accuracy preview: {acc:.4f}")
print(classification_report(y_valid, pred_valid, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_valid, pred_valid)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Fast Preview")
plt.xlabel("Prediksi")
plt.ylabel("Label asli")
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance.head(12))

plt.figure(figsize=(7, 5))
sns.barplot(data=importance.head(12), x="importance", y="feature")
plt.title("Fitur Paling Berpengaruh - Fast Preview")
plt.tight_layout()
plt.show()

## 7. Contoh Prediksi Gambar Validasi

Bagian ini menampilkan beberapa gambar validasi beserta label asli dan prediksi model. Dari sini pola kesalahan awal bisa terlihat secara visual.

In [ ]:
preview = valid_features.copy()
preview["actual"] = encoder.inverse_transform(y_valid)
preview["predicted"] = encoder.inverse_transform(pred_valid)
preview["correct"] = preview["actual"] == preview["predicted"]

display(preview[["path", "actual", "predicted", "correct"]].head(10))
print("Benar:", int(preview["correct"].sum()), "/", len(preview))

In [ ]:
sample_preview = preview.sample(min(12, len(preview)), random_state=RANDOM_STATE)
cols = 4
rows = int(np.ceil(len(sample_preview) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
axes = np.array(axes).reshape(-1)

for ax, (_, row) in zip(axes, sample_preview.iterrows()):
    image = Image.open(row["path"]).convert("RGB")
    ax.imshow(image)
    mark = "OK" if row["correct"] else "MISS"
    ax.set_title(f"{mark}\nA: {row['actual']}\nP: {row['predicted']}", fontsize=8)
    ax.axis("off")

for ax in axes[len(sample_preview):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Catatan Penggunaan

Versi cepat ini cocok untuk melihat alur dan contoh hasil awal. Jika ingin hasil yang lebih stabil:
- Naikkan TRAIN_PER_CLASS dan VALID_PER_CLASS.
- Jalankan notebook lengkap untuk segmentasi LAB, GLCM, LBP, Random Forest, dan XGBoost.
- Bandingkan confusion matrix untuk melihat kelas yang paling sering tertukar.